# Source Columns and Data Types

Lists the column names and the data types pandas infers for each source file in `data/raw/`, read as-is (no type casting).
The output is a markdown table for the **Initial File Exploration** section of the README.

Note: these are raw inferred types, not the target database types. For example, CSV date columns appear as `text`, Excel date columns as `datetime`.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

# Works from the project root or from notebooks/
RAW_DIR = next(p for p in [Path("data/raw"), Path("../data/raw")] if p.exists())
print("Reading from:", RAW_DIR.resolve())

Reading from: C:\Users\thaol\vietdist-sales-distribution-analytics\data\raw


## 2. Source files and sheets

Report sheets (`Summary` in SRC02, `Category_Summary` in SRC04) are not listed, because they are not loaded.

In [2]:
# (id, file name, sheets to read; None = CSV)
SOURCES = [
    ("SRC01", "SRC01_sales_transactions.csv", None),
    ("SRC02", "SRC02_sales_target_plan.xlsx", ["Plan_v1_Original", "Plan_v2_Adjustment_H2"]),
    ("SRC03", "SRC03_customer_master.csv", None),
    ("SRC04", "SRC04_product_master.xlsx", ["Products"]),
    ("SRC05", "SRC05_distributor_orders.xlsx", ["Orders"]),
    ("SRC06", "SRC06_distributor_master.csv", None),
    ("SRC07", "SRC07_employee_master.xlsx", ["Employee_v1_StartYear", "Employee_v2_Adjustment", "Change_Log"]),
    ("SRC08", "SRC08_territory_mapping.xlsx", ["Sheet1"]),
    ("SRC09", "SRC09_return_transactions.csv", None),
    ("SRC10", "SRC10_promotion_program.xlsx", ["Sheet1"]),
]

missing = [f for _, f, _ in SOURCES if not (RAW_DIR / f).exists()]
print("Missing files:", missing or "none")

Missing files: none


## 3. Helper functions

In [3]:
def friendly_type(dtype) -> str:
    """Map a pandas dtype to a short, readable type name."""
    return {"i": "int", "u": "int", "f": "float", "b": "bool", "M": "datetime"}.get(dtype.kind, "text")


def describe_columns(df: pd.DataFrame) -> str:
    return ", ".join(f"`{col}` {friendly_type(dt)}" for col, dt in df.dtypes.items())

## 4. Read every source and build the table

Sheets with identical columns and types (e.g. the v1 / v2 snapshots) share one row.

In [4]:
rows = []
for src_id, file_name, sheets in SOURCES:
    path = RAW_DIR / file_name
    if sheets is None:
        frames = {"–": pd.read_csv(path, encoding="utf-8")}
    else:
        frames = pd.read_excel(path, sheet_name=sheets)

    grouped = {}
    for sheet, df in frames.items():
        grouped.setdefault(describe_columns(df), []).append((sheet, df.shape[1]))

    for cols, sheet_info in grouped.items():
        sheet_label = " / ".join(f"`{s}`" if s != "–" else s for s, _ in sheet_info)
        n_cols = sheet_info[0][1]
        rows.append(f"| {src_id} | {file_name} | {sheet_label} | {n_cols} | {cols} |")

table_md = "\n".join([
    "| ID | File | Sheet | Cols | Columns (data type) |",
    "|---|---|---|---|---|",
    *rows,
])

## 5. Output

Copy the printed table into the README. The rendered version below is for a quick visual check.

In [5]:
print(table_md)

| ID | File | Sheet | Cols | Columns (data type) |
|---|---|---|---|---|
| SRC01 | SRC01_sales_transactions.csv | – | 21 | `order_id` text, `order_date` text, `order_month` int, `order_quarter` int, `order_year` int, `customer_id` text, `region` text, `province` text, `channel` text, `employee_id` text, `product_id` text, `product_category` text, `quantity` int, `unit_price` int, `discount_pct` int, `discount_amount` float, `gross_amount` int, `net_amount` float, `delivery_status` text, `payment_method` text, `payment_status` text |
| SRC02 | SRC02_sales_target_plan.xlsx | `Plan_v1_Original` / `Plan_v2_Adjustment_H2` | 13 | `plan_version` text, `version_date` text, `effective_from` text, `effective_to` text, `employee_id` text, `employee_name` text, `region` text, `team` text, `year` int, `month` int, `target_revenue` int, `target_quantity` int, `target_new_customers` int |
| SRC03 | SRC03_customer_master.csv | – | 12 | `customer_id` text, `customer_name` text, `customer_type` text, `c

In [6]:
from IPython.display import Markdown, display

display(Markdown(table_md))

| ID | File | Sheet | Cols | Columns (data type) |
|---|---|---|---|---|
| SRC01 | SRC01_sales_transactions.csv | – | 21 | `order_id` text, `order_date` text, `order_month` int, `order_quarter` int, `order_year` int, `customer_id` text, `region` text, `province` text, `channel` text, `employee_id` text, `product_id` text, `product_category` text, `quantity` int, `unit_price` int, `discount_pct` int, `discount_amount` float, `gross_amount` int, `net_amount` float, `delivery_status` text, `payment_method` text, `payment_status` text |
| SRC02 | SRC02_sales_target_plan.xlsx | `Plan_v1_Original` / `Plan_v2_Adjustment_H2` | 13 | `plan_version` text, `version_date` text, `effective_from` text, `effective_to` text, `employee_id` text, `employee_name` text, `region` text, `team` text, `year` int, `month` int, `target_revenue` int, `target_quantity` int, `target_new_customers` int |
| SRC03 | SRC03_customer_master.csv | – | 12 | `customer_id` text, `customer_name` text, `customer_type` text, `channel` text, `province` text, `region` text, `address` text, `phone` int, `tax_code` float, `join_date` text, `credit_limit` int, `status` text |
| SRC04 | SRC04_product_master.xlsx | `Products` | 10 | `product_id` text, `product_name` text, `category` text, `sub_category` text, `unit` text, `unit_price` int, `cost_price` int, `weight_gram` int, `status` text, `launch_date` text |
| SRC05 | SRC05_distributor_orders.xlsx | `Orders` | 21 | `order_id` text, `order_date` text, `order_month` int, `order_quarter` int, `distributor_id` text, `region` text, `channel` text, `product_id` text, `product_category` text, `qty_ordered` int, `qty_delivered` int, `fill_rate_pct` float, `unit_price_list` int, `distributor_price` int, `gross_amount` int, `delivered_amount` int, `expected_delivery_date` text, `actual_delivery_date` text, `ontime_delivery` text, `delivery_status` text, `payment_terms` text |
| SRC06 | SRC06_distributor_master.csv | – | 14 | `distributor_id` text, `distributor_name` text, `tier` text, `channel` text, `province` text, `region` text, `contact_person` text, `phone` int, `email` text, `tax_code` int, `join_date` text, `credit_limit` int, `status` text, `assigned_supervisor_id` text |
| SRC07 | SRC07_employee_master.xlsx | `Employee_v1_StartYear` | 15 | `employee_id` text, `full_name` text, `gender` text, `date_of_birth` text, `join_date` text, `position` text, `region` text, `team` text, `email` text, `phone` int, `status` text, `version` text, `effective_date` text, `resign_date` float, `transfer_note` float |
| SRC07 | SRC07_employee_master.xlsx | `Employee_v2_Adjustment` | 15 | `employee_id` text, `full_name` text, `gender` text, `date_of_birth` text, `join_date` text, `position` text, `region` text, `team` text, `email` text, `phone` int, `status` text, `version` text, `effective_date` text, `resign_date` text, `transfer_note` text |
| SRC07 | SRC07_employee_master.xlsx | `Change_Log` | 8 | `employee_id` text, `employee_name` text, `change_type` text, `effective_date` text, `old_region` text, `new_region` text, `old_team` text, `new_team` text |
| SRC08 | SRC08_territory_mapping.xlsx | `Sheet1` | 8 | `territory_id` text, `employee_id` text, `customer_id` text, `region` text, `team` text, `effective_date` text, `expiry_date` text, `version` text |
| SRC09 | SRC09_return_transactions.csv | – | 14 | `return_id` text, `original_order_id` text, `return_date` text, `return_month` int, `customer_id` text, `employee_id` text, `product_id` text, `region` text, `province` text, `return_quantity` int, `unit_price` int, `return_amount` int, `return_reason` text, `status` text |
| SRC10 | SRC10_promotion_program.xlsx | `Sheet1` | 14 | `promotion_id` text, `promotion_name` text, `promotion_type` text, `target_channel` text, `target_region` text, `start_date` text, `end_date` text, `applicable_products` text, `discount_pct` int, `min_order_quantity` int, `budget_vnd` int, `actual_cost_vnd` int, `status` text, `created_by` text |